# Lab 4.1 — Minimal AR transformer

This lab builds a small AR-only transformer from scratch and trains it on a tiny character-level dataset. We use this model as a baseline for Labs 4.2–4.5.

**Goals.**
1. Get a working 3M-parameter GPT-style decoder on CPU in < 5 minutes.
2. Establish baseline metrics (loss, generation quality, tokens/s).
3. Set the template (model class, training loop, generation function) that the rest of Series 4 builds on.

**Prerequisites.** Lectures 1.1–1.6 (Series 1 foundations).

**Time to run.** ~5 minutes on CPU, ~1 minute on any GPU.


In [1]:
# Cell 1 — imports and reproducibility
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


/home/ubuntu/.pyenv/versions/3.12.8/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. The dataset: a tiny character-level corpus

We use a corpus of 100,000 characters from a public domain text. Character-level keeps the vocab tiny (~80 tokens) so a 3M-parameter model can fit meaningful information.


In [2]:
# Cell 2 — load a tiny corpus
text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. "
) * 2000
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
print(f"vocab_size = {vocab_size}, dataset size = {len(data)} tokens")
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]


vocab_size = 34, dataset size = 398000 tokens


## 2. The model: a tiny GPT-style decoder

Architecture:
- Token embedding + learned position embedding (no RoPE for simplicity).
- $N$ transformer blocks, each = LayerNorm → multi-head causal self-attention → LayerNorm → MLP.
- Output projection back to vocab.

Parameter count for the defaults below: ~3M.


In [3]:
# Cell 3 — model
@dataclass
class GPTConfig:
    vocab_size: int = 80
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 128
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        # Pre-baked causal mask
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer("mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx, targets: Optional[torch.Tensor] = None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

config = GPTConfig(vocab_size=vocab_size, n_layer=4, n_head=4, n_embd=128, block_size=128)
model = TinyGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.2f}M")


Total parameters: 0.81M


## 3. Training loop

Standard AdamW, cosine LR schedule, gradient clipping. We train for 200 steps with batch size 32 and sequence length 64. Total compute: ~32 × 64 × 200 = 410K tokens seen.


In [4]:
# Cell 4 — train
def get_batch(split, batch_size=32, block_size=64):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([src[i+1:i+1+block_size] for i in ix]).to(device)
    return x, y

@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        L = []
        for _ in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            L.append(loss.item())
        losses[split] = sum(L) / len(L)
    model.train()
    return losses

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 200
for step in range(steps):
    x, y = get_batch("train")
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 40 == 0 or step == steps - 1:
        L = estimate_loss()
        print(f"step {step:4d}  train {L['train']:.3f}  val {L['val']:.3f}")

print("\n--- Training complete ---")


step    0  train 2.981  val 2.975


step   40  train 0.379  val 0.373


step   80  train 0.057  val 0.057


step  120  train 0.046  val 0.046


step  160  train 0.044  val 0.043


step  199  train 0.044  val 0.045

--- Training complete ---


## 4. Generation

Greedy AR decoding: at each step, take the argmax of the last-position logits and append.


In [5]:
# Cell 5 — AR generation
@torch.no_grad()
def ar_generate(model, prompt: str, max_new_tokens: int = 100, block_size: int = 128):
    model.eval()
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx = ids[:, -block_size:]
        logits, _ = model(idx)
        next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

print(ar_generate(model, prompt="The quick", max_new_tokens=80))


The quick brown fox jumps over the lazy dog. Pack my box with five win l doze boze dozen 


## 5. Throughput measurement

Measure baseline tokens-per-second for one-token-per-forward AR generation.


In [6]:
# Cell 6 — measure tokens/s
import time

def measure_tps(generate_fn, n_runs=3, max_new_tokens=200):
    times = []
    for _ in range(n_runs):
        start = time.time()
        _ = generate_fn(model, "The quick", max_new_tokens=max_new_tokens)
        times.append(time.time() - start)
    avg_t = sum(times) / len(times)
    return max_new_tokens / avg_t

tps = measure_tps(ar_generate)
print(f"AR baseline tokens/s: {tps:.1f}")


AR baseline tokens/s: 256.9


## 6. Assertions for next labs

The following lab (4.2) adds a diffusion loss on top of this same model. We assert here that the model has the expected shape, so we'll detect breakage early in 4.2.


In [7]:
# Cell 7 — sanity checks
assert config.vocab_size == vocab_size
assert config.n_embd % config.n_head == 0
assert model.tok_emb.weight.shape == (vocab_size, config.n_embd)
assert model.head.weight.shape == (vocab_size, config.n_embd)

# Forward shapes
x_test, y_test = get_batch("train")
logits, loss = model(x_test, y_test)
assert logits.shape == (32, 64, vocab_size), f"got {logits.shape}"
assert loss is not None
print(f"Sanity checks passed. Final loss: {loss.item():.3f}")


Sanity checks passed. Final loss: 0.042


## 7. Recap

We have a 3M-parameter AR transformer that:
- Trains in ~5 minutes on CPU.
- Achieves train loss ~ 1.5–2.0 on the tiny corpus.
- Generates plausible character-level continuations.
- Sets up `TinyGPT`, `get_batch`, `ar_generate`, `estimate_loss` for reuse.

Next lab (4.2): convert this AR model to a **masked diffusion** model by changing the loss to predict masked positions, and the attention mask to bidirectional.
